# Batch HMM Classification for Carbonic Anhydrase Sequences

This notebook processes all FASTA files in an `input_sequences` folder and classifies each sequence using pre-built HMM profiles for carbonic anhydrase families.

## Pipeline:
1. Scan `input_sequences/` for all FASTA files (*.fasta, *.fa, *.faa, *.txt)
2. For each file, run `hmmscan` against the HMM library
3. Parse results and extract best hit per sequence
4. Save results to `new_classification_results/` as CSV files

## Requirements:
- HMMER installed (hmmscan command)
- Pre-built HMM library (all_classes.hmm)
- Conda environment configured (use environment_config.sh)

## Usage Instructions

### Before Running:
1. Ensure HMMER is installed and accessible (test with `hmmscan --version`)
2. Build or locate your HMM library (`all_classes.hmm`)
3. Place FASTA files in the `input_sequences/` directory

### Running the Notebook:
1. Update paths in Section 2 if needed
2. Run all cells sequentially
3. Check `new_classification_results/` for CSV outputs

### Output Files:
- **Individual files**: `<filename>_classification.csv` - Results for each input file
- **Combined results**: `all_classifications_<timestamp>.csv` - All sequences combined
- **Processing summary**: `processing_summary_<timestamp>.csv` - Overview of batch processing

### CSV Column Descriptions:
- `seq_id`: Sequence identifier (first token of FASTA header)
- `seq_description`: Full FASTA header
- `seq_length`: Sequence length
- `predicted_class`: Predicted CA family (alpha, beta, gamma, etc., or NO_HIT)
- `predicted_model`: Full HMM model name
- `bit_score`: HMMER bit score (higher = better match)
- `evalue`: E-value (lower = more significant)
- `source_file`: Original FASTA filename

### Troubleshooting:
- If no files are found, check that FASTA files have extensions: .fasta, .fa, .faa, .fas, or .txt
- If hmmscan fails, ensure HMMER is properly installed and in your PATH
- For large datasets (>10,000 sequences), consider processing in batches
- Sequences with NO_HIT did not match any HMM profile above threshold

### Performance Notes:
- Processing time depends on: number of files, sequences per file, and sequence length
- The `--cpu 4` flag uses 4 CPU cores; adjust based on your system
- Temporary files are stored in `OUTPUT/tmp_batch_classify/`
- Use Section 8 to clean up and to save disk space


## 1. Configuration & Imports

In [ ]:
from pathlib import Path
from Bio import SeqIO
import pandas as pd
import subprocess
import shlex
import re
from datetime import datetime
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

## 2. Define Paths

In [ ]:
# Base paths - MODIFY THESE AS NEEDED
BASE = Path("~/HHM-Profiles-for-CA-Enzymes").expanduser()
REBASE = BASE / "OUTPUT"

# Input/Output directories
INPUT_DIR = BASE / "input_sequences"  # Where your FASTA files are located
OUTPUT_DIR = BASE / "new_classification_results"  # Where CSV results will be saved

# HMM library and temporary files
HMM_LIB = REBASE / "hmmer_lib" / "all_classes.hmm"
TMP_DIR = REBASE / "tmp_batch_classify"

# Create directories if they don't exist
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)

# CA family names (for reference)
CA_FAMILIES = ["alpha", "beta", "gamma", "delta", "epsilon", "zeta", "eta", "theta", "iota"]

print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"HMM library: {HMM_LIB}")
print(f"Temporary directory: {TMP_DIR}")

# Verify HMM library exists
if not HMM_LIB.exists():
    raise FileNotFoundError(f"HMM library not found at {HMM_LIB}. Please build HMMs first.")
else:
    print(f"HMM library found")

## 3. Helper Functions

In [ ]:
def first_token(s: str) -> str:
    """Extract first token from FASTA header."""
    return (s or "").split()[0]

def validate_fasta(fasta_path: Path) -> tuple[bool, int]:
    """
    Validate FASTA file and count sequences.
    Returns: (is_valid, sequence_count)
    """
    try:
        count = sum(1 for _ in SeqIO.parse(str(fasta_path), "fasta"))
        return (True, count)
    except Exception as e:
        print(f"  Warning: Could not parse {fasta_path.name}: {e}")
        return (False, 0)

def run_hmmscan(fasta_file: Path, output_prefix: str) -> tuple[Path, Path]:
    """
    Run hmmscan on a FASTA file.
    Returns paths to (tblout, domtblout) files.
    """
    tbl_out = TMP_DIR / f"{output_prefix}.tbl"
    dom_out = TMP_DIR / f"{output_prefix}.domtbl"
    
    cmd = (
        f'hmmscan --cpu 4 '
        f'--tblout "{tbl_out}" '
        f'--domtblout "{dom_out}" '
        f'"{HMM_LIB}" '
        f'"{fasta_file}"'
    )
    
    # Run hmmscan
    result = subprocess.run(
        shlex.split(cmd),
        capture_output=True,
        text=True,
        check=False
    )
    
    if result.returncode != 0:
        raise RuntimeError(f"hmmscan failed: {result.stderr}")
    
    return tbl_out, dom_out

def parse_hmmscan_results(domtbl_path: Path, fasta_file: Path) -> pd.DataFrame:
    """
    Parse hmmscan domtbl output and return best hit per sequence.
    
    Returns DataFrame with columns:
    - seq_id: Sequence identifier
    - seq_description: Full FASTA header
    - seq_length: Sequence length
    - predicted_class: Predicted CA family (alpha, beta, etc.)
    - predicted_model: Full model name
    - bit_score: HMMER bit score
    - evalue: E-value
    - source_file: Original FASTA filename
    """
    # Column names for domtblout format
    dom_cols = [
        "target", "tacc", "tlen", "query", "qacc", "qlen",
        "fs_evalue", "fs_score", "fs_bias",
        "num", "of", "dom_cevalue", "dom_ievalue", "dom_score", "dom_bias",
        "hmmfrom", "hmmto", "alifrom", "alito", "envfrom", "envto",
        "acc", "desc"
    ]
    
    # Read domtbl file
    try:
        dom_hits = pd.read_csv(
            domtbl_path,
            sep=r"\s+",
            comment="#",
            header=None,
            names=dom_cols,
            usecols=list(range(23)),
            engine="python"
        )
    except Exception as e:
        print(f"  Warning: Could not parse {domtbl_path.name}: {e}")
        return pd.DataFrame()
    
    if len(dom_hits) == 0:
        print(f"  Warning: No hits found in {domtbl_path.name}")
        return pd.DataFrame()
    
    # Rename and select relevant columns
    hits = dom_hits.rename(columns={
        "target": "predicted_model",
        "query": "seq_id",
        "qlen": "seq_length",
        "fs_score": "bit_score",
        "fs_evalue": "evalue"
    })[["seq_id", "seq_length", "predicted_model", "bit_score", "evalue"]]
    
    # Extract predicted class from model name (e.g., "8ca.alpha.45" -> "alpha")
    hits["predicted_class"] = hits["predicted_model"].str.extract(r'^8ca\.([a-z]+)\.')
    
    # Get best hit per sequence (highest bit score)
    best_hits = (
        hits.sort_values(["seq_id", "bit_score"], ascending=[True, False])
        .groupby("seq_id", as_index=False)
        .first()
    )
    
    # Load original FASTA to get full descriptions
    seq_info = {}
    for record in SeqIO.parse(str(fasta_file), "fasta"):
        seq_id = first_token(record.id or record.description)
        seq_info[seq_id] = {
            "seq_description": record.description,
            "seq_length_fasta": len(record.seq)
        }
    
    # Merge with sequence info
    seq_info_df = pd.DataFrame.from_dict(seq_info, orient="index").reset_index()
    seq_info_df.columns = ["seq_id", "seq_description", "seq_length_fasta"]
    
    result = best_hits.merge(seq_info_df, on="seq_id", how="left")
    
    # Add source file
    result["source_file"] = fasta_file.name
    
    # Reorder columns for clarity
    result = result[[
        "seq_id", "seq_description", "seq_length",
        "predicted_class", "predicted_model",
        "bit_score", "evalue", "source_file"
    ]]
    
    return result

def process_sequences_without_hits(fasta_file: Path, classified_ids: set) -> pd.DataFrame:
    """
    Create records for sequences that had no HMM hits.
    """
    no_hit_records = []
    
    for record in SeqIO.parse(str(fasta_file), "fasta"):
        seq_id = first_token(record.id or record.description)
        if seq_id not in classified_ids:
            no_hit_records.append({
                "seq_id": seq_id,
                "seq_description": record.description,
                "seq_length": len(record.seq),
                "predicted_class": "NO_HIT",
                "predicted_model": "NO_HIT",
                "bit_score": None,
                "evalue": None,
                "source_file": fasta_file.name
            })
    
    return pd.DataFrame(no_hit_records)

print("Helper functions defined")

## 4. Scan Input Directory for FASTA Files

In [ ]:
# Find all FASTA files
fasta_patterns = ["*.fasta", "*.fa", "*.faa", "*.fas", "*.txt"]
fasta_files = []

for pattern in fasta_patterns:
    fasta_files.extend(INPUT_DIR.glob(pattern))

# Remove duplicates and sort
fasta_files = sorted(set(fasta_files))

print(f"Found {len(fasta_files)} FASTA file(s) in {INPUT_DIR}:")

# Validate files and count sequences
valid_files = []
total_sequences = 0

for fasta_file in fasta_files:
    is_valid, seq_count = validate_fasta(fasta_file)
    if is_valid and seq_count > 0:
        valid_files.append(fasta_file)
        total_sequences += seq_count
        print(f" {fasta_file.name}: {seq_count} sequences")
    else:
        print(f" {fasta_file.name}: Invalid or empty")

print(f"\nTotal valid files: {len(valid_files)}")
print(f"Total sequences to classify: {total_sequences}")

if len(valid_files) == 0:
    raise ValueError(f"No valid FASTA files found in {INPUT_DIR}. Please add FASTA files to process.")

## 5. Process Each FASTA File

In [ ]:
# Store all results
all_results = []
processing_summary = []

print("\nProcessing files...\n")

for i, fasta_file in enumerate(tqdm(valid_files, desc="Processing FASTA files"), 1):
    print(f"\n[{i}/{len(valid_files)}] Processing: {fasta_file.name}")
    
    try:
        # Generate unique output prefix
        output_prefix = f"{fasta_file.stem}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        
        # Run hmmscan
        print("  Running hmmscan...")
        tbl_out, dom_out = run_hmmscan(fasta_file, output_prefix)
        
        # Parse results
        print("  Parsing results...")
        results = parse_hmmscan_results(dom_out, fasta_file)
        
        if len(results) == 0:
            print("  Warning: No hits found for any sequences")
            continue
        
        # Check for sequences without hits
        classified_ids = set(results["seq_id"])
        no_hit_df = process_sequences_without_hits(fasta_file, classified_ids)
        
        # Combine results
        combined_results = pd.concat([results, no_hit_df], ignore_index=True)
        all_results.append(combined_results)
        
        # Save individual file results
        output_csv = OUTPUT_DIR / f"{fasta_file.stem}_classification.csv"
        combined_results.to_csv(output_csv, index=False)
        
        # Summary stats
        n_total = len(combined_results)
        n_classified = len(results)
        n_no_hit = len(no_hit_df)
        
        print(f"  Classified: {n_classified}/{n_total} sequences")
        if n_no_hit > 0:
            print(f"    No hits: {n_no_hit} sequences")
        print(f"  Saved to: {output_csv.name}")
        
        # Store summary
        processing_summary.append({
            "file": fasta_file.name,
            "total_sequences": n_total,
            "classified": n_classified,
            "no_hits": n_no_hit,
            "output_file": output_csv.name
        })
        
    except Exception as e:
        print(f"  Error processing {fasta_file.name}: {e}")
        processing_summary.append({
            "file": fasta_file.name,
            "total_sequences": 0,
            "classified": 0,
            "no_hits": 0,
            "output_file": f"ERROR: {str(e)}"
        })

print("\n" + "="*60)
print("Processing complete!")
print("="*60)

## 6. Create Combined Results and Summary

In [ ]:
if len(all_results) > 0:
    # Combine all results into single DataFrame
    combined_df = pd.concat(all_results, ignore_index=True)
    
    # Save combined results
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    combined_output = OUTPUT_DIR / f"all_classifications_{timestamp}.csv"
    combined_df.to_csv(combined_output, index=False)
    
    print(f"\n Combined results saved to: {combined_output.name}")
    print(f"  Total sequences classified: {len(combined_df)}")
    
    # Create processing summary
    summary_df = pd.DataFrame(processing_summary)
    summary_output = OUTPUT_DIR / f"processing_summary_{timestamp}.csv"
    summary_df.to_csv(summary_output, index=False)
    
    print(f"\n Processing summary saved to: {summary_output.name}")
    print("\nProcessing Summary:")
    display(summary_df)
    
else:
    print("\n No results to combine.")

## 7. Classification Statistics

In [ ]:
if len(all_results) > 0:
    print("\nClassification Statistics:")
    print("="*60)
    
    # Overall class distribution
    class_counts = combined_df["predicted_class"].value_counts()
    print("\nPredicted Class Distribution:")
    for class_name, count in class_counts.items():
        percentage = (count / len(combined_df)) * 100
        print(f"  {class_name:12s}: {count:5d} ({percentage:5.2f}%)")
    
    # Statistics by file
    print("\nClassification by Source File:")
    file_stats = combined_df.groupby(["source_file", "predicted_class"]).size().unstack(fill_value=0)
    display(file_stats)
    
    # Score statistics (for sequences with hits)
    hits_only = combined_df[combined_df["predicted_class"] != "NO_HIT"]
    if len(hits_only) > 0:
        print("\nBit Score Statistics (sequences with hits):")
        score_stats = hits_only.groupby("predicted_class")["bit_score"].describe()
        display(score_stats.round(2))
    
    print("\n" + "="*60)
    print(f"Results saved to: {OUTPUT_DIR}")
    print("="*60)

## 8. Cleanup

In [ ]:

import shutil
if TMP_DIR.exists():
    shutil.rmtree(TMP_DIR)
    print(f" Cleaned up temporary directory: {TMP_DIR}")

print("\nNote: Temporary files are kept in:", TMP_DIR)
